# 03. Target Variable Construction 

This notebook builds the count-based target variable for each hexagon: how many popular cafes fall inside it. This uses the `is_popular_prov` cafes from
`cafes_clean.csv` (from notebook 02) and the hexagon grid from `berlin_h3_res8.geojson` (from notebook 01).

Hexagons with zero popular cafes are kept as 0, not dropped since they are real observations for the Poisson count model used later.

**Output:** the hexagon grid with a new `popular_cafe_count` column, saved to `data/processed/berlin_h3_res8_with_target.geojson`.

In [9]:

import pandas as pd
import geopandas as gpd
from pathlib import Path
import sys
sys.path.append("../src")
from target_variable import add_hex_id, count_cafes_per_hex
 


In [10]:
GRID = Path("../data/external/berlin_h3_res8.geojson")
CAFES = Path("../data/processed/cafes_clean.csv")
 
grid = gpd.read_file(GRID)
cafes = pd.read_csv(CAFES)

## Keep only the popular cafes

`cafes_clean.csv` contains all 2,434 cleaned cafes, not just the popular ones. Only cafes flagged `is_popular_prov` (rating >= 4.2 and >= 100 reviews) are used to build the target.

In [11]:
popular_cafes = cafes[cafes["is_popular_prov"] == 1].copy()
print(f"Popular cafes: {len(popular_cafes)}")

Popular cafes: 923


## Assign each popular cafe to a hexagon

For every cafe, `h3.latlng_to_cell()` finds which resolution-8 hexagon contains its coordinates.

In [12]:
popular_cafes = add_hex_id(popular_cafes, lat_col="location_lat", lng_col="location_lng", resolution=grid["resolution"].iloc[0])
popular_cafes[["location_lat", "location_lng", "h3_index"]].head()

,location_lat,location_lng,h3_index
10,52.534274,13.431518,881f1d4f27fffff
25,52.509096,13.374804,881f1d48bdfffff
40,52.343900,12.982718,881f188191fffff
43,52.511484,13.426863,881f1d4d61fffff
49,52.457916,13.577347,881f18b709fffff


In [13]:
grid = count_cafes_per_hex(popular_cafes, grid, hex_id_col="h3_index")

print("Total popular cafes assigned:", grid["popular_cafe_count"].sum())
print("Hexagons with at least 1 cafe:", (grid["popular_cafe_count"] > 0).sum())
print("Hexagons with zero cafes:", (grid["popular_cafe_count"] == 0).sum())
grid["popular_cafe_count"].describe()

Total popular cafes assigned: 844
Hexagons with at least 1 cafe: 268
Hexagons with zero cafes: 1085


count    1353.000000
mean        0.623799
std         2.012813
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max        21.000000
Name: popular_cafe_count, dtype: float64

## Check the count distribution




In [14]:
grid["popular_cafe_count"].value_counts().sort_index()

popular_cafe_count
0     1085
1      128
2       43
3       24
4       17
5       14
6       10
7        5
8        3
9        7
10       3
11       1
12       3
13       1
14       2
15       2
16       2
18       1
19       1
21       1
Name: count, dtype: int64

In [15]:
OUT = Path("../data/processed/berlin_h3_res8_with_target.geojson")
grid.to_file(OUT, driver="GeoJSON")
print(f"Saved grid with target -> {OUT}")

Saved grid with target -> ..\data\processed\berlin_h3_res8_with_target.geojson


confirm the data is zero-inflated and low-count

## Check for cafes that didn't match a hexagon

some cafes may sit just outside Berlin's administrative boundary. This checks how many popular cafes were not matched to a hexagon during the merge and inspects their coordinates 

In [16]:
# which popular cafes didn't get matched to a hexagon?
matched_ids = set(grid["h3_index"])
missing = popular_cafes[~popular_cafes["h3_index"].isin(matched_ids)]

print(f"Unmatched popular cafes: {len(missing)}")
missing[["title", "location_lat", "location_lng", "h3_index"]]

Unmatched popular cafes: 79


,title,location_lat,location_lng,h3_index
40,Café Traumschaum,52.343900,12.982718,881f188191fffff
92,Café Mühle,52.679955,13.586370,881f1d473bfffff
110,Päuschen Softeis & Kaffee,52.646321,13.521260,881f1d4097fffff
162,Bäckerei Schneider GmbH,52.372627,13.615873,881f18b537fffff
204,Kaffeehaus Madlen,52.649281,13.517337,881f1d4095fffff
...,...,...,...,...
2326,Café Midi – Das Café für Kleine und Große im T...,52.407380,13.065340,881f188729fffff
2356,Café Heimath,52.347160,12.987066,881f18819bfffff
2367,Anni's Café,52.549125,13.635323,881f1d4cd7fffff
2371,Café Freundlich – Auf dem Telegrafenberg in Po...,52.381915,13.064974,881f188093fffff


**Finding:** from a quick search, the 79 unmatched cafes cluster at the edges of the search area eg in Potsdam, Schönefeld, and Brandenburg. These cafes fall outside Berlin's administrative boundary. Excluding them enforces the study's scope (Berlin only)
**Final target variable:** 844 popular cafes assigned across 268 hexagons.